<a href="https://colab.research.google.com/github/mdsadaqathali/week3/blob/main/w3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
print("TensorFlow Version:",tf.__version__)
print("GPU Available:",tf.config.list_physical_devices("GPU"))
train_dir = "train"
val_dir = "val"
test_dir = "test"
print("Train Cats:",len(os.listdir("train/cats")))
print("Train Dogs:",len(os.listdir("train/dogs")))
print("Validation Cats:",len(os.listdir("val/cats")))
print("Validation Dogs:",len(os.listdir("val/dogs")))
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=20
)
val_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode="binary"
)
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode="binary"
)
print("Class Names:",train_generator.class_indices)
model = Sequential()
model.add(Conv2D(32,(3,3),activation="relu",input_shape=(150,150,3)))
model.add(MaxPooling2D(2,2))
model.add(Conv2D(64,(3,3),activation="relu"))
model.add(MaxPooling2D(2,2))
model.add(Conv2D(128,(3,3),activation="relu"))
model.add(MaxPooling2D(2,2))
model.add(Flatten())
model.add(Dense(512,activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(1,activation="sigmoid"))
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
print(model.summary())
checkpoint = ModelCheckpoint(
    "best_cat_dog_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[checkpoint]
)
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"],label="Training Accuracy")
plt.plot(history.history["val_accuracy"],label="Validation Accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"],label="Training Loss")
plt.plot(history.history["val_loss"],label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)
test_loss,test_accuracy = model.evaluate(test_generator)
print("Test Loss:",test_loss)
print("Test Accuracy:",test_accuracy)
loaded_model = load_model("best_cat_dog_model.keras")
images,labels = next(test_generator)
predictions = loaded_model.predict(images)
plt.figure(figsize=(12,12))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    prediction = "Dog" if predictions[i][0] >= 0.5 else "Cat"
    actual = "Dog" if labels[i] == 1 else "Cat"
    plt.title("Predicted: "+prediction+"\nActual: "+actual)
    plt.axis("off")
plt.tight_layout()
plt.show()
def predict_image(image_path):
    image = tf.keras.utils.load_img(image_path,target_size=(150,150))
    image_array = tf.keras.utils.img_to_array(image)
    image_array = image_array / 255.0
    image_array = np.expand_dims(image_array,axis=0)
    prediction = loaded_model.predict(image_array)[0][0]
    if prediction >= 0.5:
        result = "Dog"
        probability = prediction * 100
    else:
        result = "Cat"
        probability = (1-prediction) * 100
    print("Image:",image_path)
    print("Prediction:",result)
    print("Probability:",round(probability,2),"%")
predict_image("cat.jpg")
predict_image("dog.jpg")